# Experimento multi-base — OES/DES/SES em 8 bases

Aplica **o mesmo pipeline** do experimento do artigo (pool da Tabela 3 →
SES-GA com fitness de acurácia+diversidade → DES → OES → 5 métricas → Wilcoxon)
a **8 bases**, em dois modos de pré-processamento (`baseline` e `melhorado`).
É o mesmo tratamento feito para Finnish e Maxwell, generalizado, com o
pré-processamento identificado automaticamente por base.

> O notebook do **artigo original** (apenas Finnish + Maxwell, replicação fiel)
> é o `01_replicacao_OES.ipynb` — este aqui é separado.

Seis bases são de **estimação de esforço de software**; **debutanizer** e
**abalone** foram incluídas a pedido para avaliar os 3 métodos, mas **não são**
de esforço — são regressão genérica, e servem de contraste (nelas os 3 métodos
atingem sMAPE baixo porque o problema é mais fácil).

Métricas sempre no **esforço bruto** (alvo log-transformado só no treino quando
assimétrico). Para SES/DES/OES: menor sMAPE/MRE/MASE é melhor; maior NSE/COD.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
_here = Path.cwd(); ROOT = _here if (_here / "src").exists() else _here.parent
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd
from IPython.display import Image, display
import data_multi as dm
import run_experiment_multi as rm
import figures as fg
print("Setup OK. Bases:", dm.DATASETS)

## Fontes e categoria de cada base (sem duplicatas)

Nenhuma base é duplicata das outras. As fontes:

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.DataFrame([{"Base": n, "Categoria": dm.CATEGORIA[n], "Fonte": dm.FONTE[n]}
              for n in dm.DATASETS])

## Fase 1 — Pré-processamento identificado por base

Linhas válidas, nº de features `baseline`→`melhorado`, se o alvo foi
log-transformado, e o que as regras automáticas removeram (constantes,
variância≈0, correlação>0.98, informação mútua≈0). IDs, datas, strings e
**vazamento** (ex.: China `PDR_*`/`N_effort`, `Duration`/`Length`/`Time`,
COCOMO `months`) saem nos dois modos.

In [ ]:
recs = []
for n in dm.DATASETS:
    db = dm.build(n, "baseline"); dv = dm.build(n, "melhorado"); p = dv["prep"]
    drops = "; ".join(filter(None, [
        ("const: " + ",".join(p["drop_const"])) if p["drop_const"] else "",
        ("var≈0: " + ",".join(p["drop_nzv"])) if p["drop_nzv"] else "",
        ("corr>0.98: " + ",".join(p["drop_corr"])) if p["drop_corr"] else "",
        ("MI≈0: " + ",".join(p["drop_mi"])) if p["drop_mi"] else ""])) or "—"
    recs.append({"Base": n, "Linhas": p["n_linhas"], "Feats base": db["n_features"],
                 "Feats melh": dv["n_features"], "log alvo": p["log_alvo"],
                 "Removidos no melhorado": drops})
pd.DataFrame(recs)

## Fases 2–3 — pipeline completo (holdout, seed 42)

Roda pool → SES-GA → DES → OES nos dois modos, nas 8 bases. Resumo do **OES**
(baseline vs melhorado). *(Leva ~1 min; abalone tem 4177 linhas.)*

In [ ]:
res = {n: {md: rm.rodar(n, md) for md in ["baseline", "melhorado"]} for n in dm.DATASETS}
recs = []
for n in dm.DATASETS:
    for md in ["baseline", "melhorado"]:
        o = res[n][md]["rows"]["Proposed"]
        recs.append({"Base": n, "Modo": md, **{m: round(o[m], 3) for m in rm.MC}})
pd.DataFrame(recs)

Comparação dos **15 modelos** (baseline → melhorado, com Δ) fica em
`outputs/multi/tables/<base>_compare.csv`. Abaixo, só os três ensembles de uma
base (troque `BASE`):

In [ ]:
BASE = "coc81"
p = ROOT / "outputs" / "multi" / "tables" / f"{BASE}_compare.csv"
if not p.exists():
    rm.holdout()
df = pd.read_csv(p)
df[df["Model"].isin(["Static", "DES", "Proposed"])]

## O OES (Proposto) é o melhor?

Posição do OES entre os 15 modelos, por base e modo.

In [ ]:
rk = []
for n in dm.DATASETS:
    for md in ["baseline", "melhorado"]:
        r = rm.rank_oes(res[n][md]["rows"])
        rk.append({"Base": n, "Modo": md,
                   "OES é #1 em": f"{sum(1 for m in rm.MC if r[m][0]=='Proposed')}/5",
                   "rank mediano OES": int(np.median([r[m][2] for m in rm.MC])),
                   "vence sMAPE": r["sMAPE"][0]})
pd.DataFrame(rk)

## Teste de hipótese: de H0 para H1 (quem é quem)

O Wilcoxon do artigo (Tabela 6) compara **predito × real** de cada modelo — não
diz qual método vence. Para decidir **quem é melhor** fazemos um teste **pareado
método × método** sobre os erros absolutos por instância, em dois passos:

1. **H0**: os dois métodos têm o mesmo desempenho (mediana das diferenças de
   erro = 0). **H1 (bilateral)**: diferem. Calcula-se o p-valor de Wilcoxon
   pareado.
2. **Se p ≥ 0,05** → não rejeita H0 → *empate* (não dá para eleger vencedor).
   **Se p < 0,05** → rejeita H0; aí a **direção** (método com menor erro absoluto
   mediano) define a H1 específica *‘A erra menos que B’* — é isso que responde
   **quem é quem**.

Fazemos as 3 comparações par a par (SES×DES, SES×OES, DES×OES).
⚠️ **Significância ≠ relevância**: em bases grandes (debutanizer, abalone)
diferenças minúsculas já rejeitam H0; em bases pequenas (COCOMO81 ~19 pontos de
teste) nem diferenças reais atingem significância — por isso o ranking por erro
acompanha o p-valor.

In [ ]:
sig = []
for n in dm.DATASETS:
    ens_te = res[n]["melhorado"]["ens_te"]; y = res[n]["melhorado"]["d"]["y_test_raw"]
    ordem = " > ".join(rm.ranking_metodos(ens_te, y))   # menor erro -> maior
    for r in rm.comparar_pares(ens_te, y):
        sig.append({"Base": n, "Cat": dm.CATEGORIA[n], "Ranking (menor erro→)": ordem, **r})
pd.DataFrame(sig)

## Scatter SES/DES/OES por base (modo melhorado)

O OES fica entre SES e DES **por construção** (é a média dos dois). Note os
pontos longe da diagonal — o erro relativo alto nas bases de SEE é generalizado,
não só outliers.

In [ ]:
FIG = ROOT / "outputs" / "multi" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
for n in dm.DATASETS:
    d = res[n]["melhorado"]["d"]; path = FIG / f"{n}_scatter.png"
    fg.scatter_fig(str(path), f"{n} (melhorado) — Scatter SES/DES/OES",
                   res[n]["melhorado"]["ens_tr"], res[n]["melhorado"]["ens_te"],
                   d["y_train_raw"], d["y_test_raw"])
    display(Image(str(path)))

## Robustez — CV repetida (mediana ± desvio), OES baseline vs melhorado

Mediana e desvio sobre 5 splits estratificados (a mediana é robusta a partições
catastróficas). Bases grandes foram subamostradas (cap=700) só na robustez.
Carregamos o resultado já calculado; para **regenerar** rode
`rm.robustez()` (e `rm.robustez(subset=["debutanizer","abalone"])` para as
grandes), depois `rm._append_robustez_report()`.

In [ ]:
rob = ROOT / "outputs" / "multi" / "tables" / "robustez_oes.csv"
if not rob.exists():
    rm.robustez()
pd.read_csv(rob)

## Leitura dos resultados

- **sMAPE e MRE melhoram com o pré-processamento em (quase) todas as bases**
  (efeito robusto do `log1p` + remoção de ruído/redundância); COCOMO81 é o maior
  ganho. **NSE/COD no esforço bruto são instáveis** em bases de cauda pesada
  (China, Kitchenham) — daí a mediana em CV repetida ser a leitura correta.
- **Contraste SEE × não-SEE:** abalone e debutanizer atingem sMAPE ~15–28%
  (features fortes), enquanto as bases de SEE ficam em ~35–95%. Ou seja, o sMAPE
  alto em SEE é **intrínseco à dificuldade de estimar esforço**, não do método.
- **O OES não é o melhor em nenhuma base.** Com os dados limpos, um único modelo
  forte (com frequência o **Extra Trees**) domina e o OES cai no ranking. No
  teste pareado, a maioria dos pares de SEE dá **empate** (teste pequeno); quando
  há diferença, raramente o OES é o vencedor. Tudo reforça que o resultado
  depende dos **dados e do protocolo**, não da arquitetura de ensemble.